# DeepSpeech2 — Day 1: Training Data

Goal: stage the **LibriSpeech** corpus into Azure Blob Storage so later notebooks can stream it for training.

**Pipeline per split:** download tarball → extract → upload to Blob via `azcopy` → verify (file count + total bytes) → cleanup local files.


## Task 1 — Create the Azure Blob container

Create a container named **`librispeech-raw`** in your Azure Storage account (one-time, done outside this notebook via the Azure portal or CLI). All splits will be uploaded under this container as `librispeech-raw/<split>/...`.


## Task 2 — Download dataset → Colab → Blob Storage

For each split: pull the tarball from OpenSLR into the Colab VM, extract, push to Azure Blob, verify the upload, then free the local disk so the next split fits.

### Step 1 — Install dependencies and `azcopy`

Installs the Azure SDKs we need and downloads the `azcopy` binary if it's not already on `PATH`. `azcopy` is what we use to push the extracted audio to Blob Storage (much faster than the Python SDK for many small files).


### Step 2 — `LibriSpeechPipeline` class

End-to-end pipeline for a single split. Each method is idempotent (safe to re-run) and emits timestamped progress logs.

| Method | Responsibility |
| --- | --- |
| `download` | Stream the tarball from OpenSLR to local disk. 60s socket timeout; retries up to 3× with 30s backoff. |
| `extract` | Untar with `filter="data"` to block path-traversal entries. |
| `upload_to_azure` | Push extracted files to Blob via `azcopy` using a 12h container SAS. Retries up to 3× with 30s backoff. |
| `verify` | Compare local vs remote **file count AND total bytes** — both must match. |
| `cleanup` | Delete the tarball and extracted directory; report total bytes freed. |
| `run` | Orchestrate the full sequence; skips cleanup if verify fails. |


In [ ]:
# Install Azure SDKs + helpers, then ensure the `azcopy` binary is on PATH.
# Colab images don't ship with azcopy, so we fetch the Linux build on first run.
!pip install -q azure-storage-blob azure-identity azure-keyvault-secrets python-dotenv tqdm

import shutil, subprocess

if shutil.which("azcopy") is None:
    print("azcopy not found — installing...")
    !wget -q https://aka.ms/downloadazcopy-v10-linux -O /tmp/azcopy.tar.gz
    !tar -xf /tmp/azcopy.tar.gz -C /tmp
    !cp /tmp/azcopy_linux_amd64_*/azcopy /usr/local/bin/azcopy
    !chmod +x /usr/local/bin/azcopy

print("azcopy:", shutil.which("azcopy"))
print(subprocess.run(["azcopy", "--version"], capture_output=True, text=True).stdout.strip())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 6.4 MB/s eta 0:00:00
azcopy not found — installing...
azcopy: /usr/local/bin/azcopy
azcopy version 10.32.2


In [ ]:
import os
import shutil
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path
from datetime import datetime, timedelta, timezone

from azure.storage.blob import (
    BlobServiceClient,
    generate_container_sas,
    ContainerSasPermissions,
)


class LibriSpeechPipeline:
    """Download LibriSpeech splits, extract, push to Azure Blob, verify, clean up."""

    # OpenSLR mirrors for each LibriSpeech split we care about.
    SPLITS = {
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
        "train-other-500": "https://www.openslr.org/resources/12/train-other-500.tar.gz",
        "dev-clean":       "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "test-clean":      "https://www.openslr.org/resources/12/test-clean.tar.gz",
    }

    def __init__(self, work_dir: str = "/content/librispeech"):
        self.work_dir = Path(work_dir)
        self.work_dir.mkdir(parents=True, exist_ok=True)

    # ---------- helpers ----------

    def _log(self, split: str, msg: str) -> None:
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] [{split}] {msg}", flush=True)

    def _tar_path(self, split: str) -> Path:
        return self.work_dir / f"{split}.tar.gz"

    def _extract_root(self, split: str) -> Path:
        return self.work_dir / split

    def _extracted_split_dir(self, split: str) -> Path:
        # Tarballs unpack to <root>/LibriSpeech/<split>/...
        return self._extract_root(split) / "LibriSpeech" / split

    def _assert_split(self, split: str) -> None:
        if split not in self.SPLITS:
            raise ValueError(f"unknown split '{split}'. valid: {list(self.SPLITS)}")

    def _dir_size(self, path: Path) -> int:
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

    # ---------- pipeline stages ----------

    def download(self, split: str) -> Path:
        """Stream the split tarball to local disk. Retries 3× on failure."""
        self._assert_split(split)
        url = self.SPLITS[split]
        dest = self._tar_path(split)

        # Idempotent: skip if a non-empty tarball is already on disk.
        if dest.exists() and dest.stat().st_size > 0:
            self._log(split, f"tarball already present at {dest} ({dest.stat().st_size / 1e9:.2f} GB) — skipping download")
            return dest

        for attempt in range(1, 4):
            try:
                self._log(split, f"downloading {url} (attempt {attempt}/3)")
                tmp = dest.with_suffix(dest.suffix + ".part")
                start = time.time()

                # 60s socket timeout protects us against silently stalled connections.
                with urllib.request.urlopen(url, timeout=60) as resp, open(tmp, "wb") as f:
                    total = int(resp.headers.get("Content-Length", 0))
                    read = 0
                    chunk = 1 << 20  # 1 MiB
                    last_report = 0
                    while True:
                        buf = resp.read(chunk)
                        if not buf:
                            break
                        f.write(buf)
                        read += len(buf)
                        # Throttle progress logs to roughly every 250 MiB.
                        if total and read - last_report > 250 * chunk:
                            pct = 100.0 * read / total
                            self._log(split, f"  ...{read / 1e9:.2f} / {total / 1e9:.2f} GB ({pct:.1f}%)")
                            last_report = read
                    # Final 100% line so the log doesn't look stalled at the end.
                    if total:
                        self._log(split, f"  ...{read / 1e9:.2f} / {total / 1e9:.2f} GB (100.0%)")

                # Atomic rename so a partial file is never mistaken for a complete one.
                tmp.rename(dest)
                elapsed = time.time() - start
                self._log(split, f"downloaded {dest.stat().st_size / 1e9:.2f} GB in {elapsed:.0f}s")
                return dest
            except Exception as e:
                self._log(split, f"download attempt {attempt} failed: {e}")
                if attempt == 3:
                    raise
                self._log(split, "waiting 30s before retry")
                time.sleep(30)

    def extract(self, split: str) -> Path:
        """Untar the split. Uses `filter='data'` to block path-traversal entries."""
        self._assert_split(split)
        tar = self._tar_path(split)
        if not tar.exists():
            raise FileNotFoundError(f"tarball missing for {split}: {tar}")

        out_root = self._extract_root(split)
        extracted_dir = self._extracted_split_dir(split)
        if extracted_dir.exists() and any(extracted_dir.iterdir()):
            self._log(split, f"already extracted at {extracted_dir} — skipping")
            return extracted_dir

        out_root.mkdir(parents=True, exist_ok=True)
        self._log(split, f"extracting {tar} -> {out_root}")
        start = time.time()
        with tarfile.open(tar, "r:gz") as tf:
            # filter="data" rejects absolute paths, .. traversal, devices, links — PEP 706 hardening.
            tf.extractall(out_root, filter="data")
        self._log(split, f"extracted in {time.time() - start:.0f}s")
        return extracted_dir

    def upload_to_azure(self, split: str, connection_string: str, container_name: str) -> None:
        """Push the extracted split to Blob Storage via azcopy. Retries 3× on failure."""
        self._assert_split(split)
        src = self._extracted_split_dir(split)
        if not src.exists():
            raise FileNotFoundError(f"extracted folder missing for {split}: {src}")

        for attempt in range(1, 4):
            try:
                # Mint a short-lived container SAS so azcopy can authenticate.
                svc = BlobServiceClient.from_connection_string(connection_string)
                account_key = svc.credential.account_key
                sas = generate_container_sas(
                    account_name=svc.account_name,
                    container_name=container_name,
                    account_key=account_key,
                    permission=ContainerSasPermissions(read=True, write=True, list=True, create=True, add=True),
                    expiry=datetime.now(timezone.utc) + timedelta(hours=12),
                )
                dest_url = f"https://{svc.account_name}.blob.core.windows.net/{container_name}/{split}?{sas}"

                self._log(split, f"uploading {src} -> blob container '{container_name}'/{split} (attempt {attempt}/3)")
                start = time.time()
                result = subprocess.run(
                    ["azcopy", "copy", f"{src}/*", dest_url, "--recursive=true", "--overwrite=ifSourceNewer"],
                    capture_output=True,
                    text=True,
                )
                if result.returncode != 0:
                    self._log(split, "azcopy FAILED")
                    print(result.stdout)
                    print(result.stderr)
                    raise RuntimeError(f"azcopy upload failed for {split}")
                self._log(split, f"upload complete in {time.time() - start:.0f}s")
                return
            except Exception as e:
                self._log(split, f"upload attempt {attempt} failed: {e}")
                if attempt == 3:
                    raise
                self._log(split, "waiting 30s before retry")
                time.sleep(30)

    def verify(self, split: str, connection_string: str, container_name: str) -> bool:
        """Cross-check that local and remote agree on both file count and total bytes."""
        self._assert_split(split)
        src = self._extracted_split_dir(split)
        if not src.exists():
            raise FileNotFoundError(f"extracted folder missing for {split}: {src}")

        local_files = [p for p in src.rglob("*") if p.is_file()]
        local_count = len(local_files)
        local_bytes = sum(p.stat().st_size for p in local_files)
        self._log(split, f"local file count: {local_count}, total bytes: {local_bytes}")

        svc = BlobServiceClient.from_connection_string(connection_string)
        container = svc.get_container_client(container_name)
        prefix = f"{split}/"
        remote_count = 0
        remote_bytes = 0
        for blob in container.list_blobs(name_starts_with=prefix):
            remote_count += 1
            remote_bytes += blob.size
        self._log(split, f"remote file count under '{prefix}': {remote_count}, total bytes: {remote_bytes}")

        # Both counts AND byte totals must match — catches truncated uploads that count would miss.
        ok = local_count == remote_count and local_bytes == remote_bytes
        if ok:
            self._log(split, "verify OK ✔")
        else:
            self._log(split, f"verify MISMATCH ✘ (local={local_count}/{local_bytes}B, remote={remote_count}/{remote_bytes}B)")
        return ok

    def cleanup(self, split: str) -> None:
        """Delete tarball + extracted dir for this split and report total bytes freed."""
        self._assert_split(split)
        tar = self._tar_path(split)
        extracted = self._extract_root(split)
        freed = 0

        # Measure sizes BEFORE deleting so the freed total reflects reality.
        if tar.exists():
            freed += tar.stat().st_size
        if extracted.exists():
            freed += self._dir_size(extracted)

        if tar.exists():
            tar.unlink()
            self._log(split, f"deleted {tar}")
        if extracted.exists():
            shutil.rmtree(extracted, ignore_errors=True)
            self._log(split, f"deleted {extracted}")

        self._log(split, f"freed ~{freed / 1e9:.2f} GB")

    def run(self, split: str, connection_string: str, container_name: str) -> None:
        """Full pipeline for one split. Skips cleanup if verify fails so the data isn't lost."""
        self._assert_split(split)
        self._log(split, "=== START ===")
        self.download(split)
        self.extract(split)
        self.upload_to_azure(split, connection_string, container_name)
        ok = self.verify(split, connection_string, container_name)
        if not ok:
            self._log(split, "skipping cleanup because verify failed")
            return
        self.cleanup(split)
        self._log(split, "=== DONE ===")


### Step 3 — Authenticate and run the pipeline

Load Azure service-principal credentials from `.env`, fetch the storage connection string from Key Vault (so the secret never lives in the notebook), then run the pipeline over every LibriSpeech split.

**Required environment variables in `.env`:**

- `AZURE_CLIENT_ID` — service principal app ID
- `AZURE_TENANT_ID` — Azure AD tenant ID
- `AZURE_CLIENT_SECRET` — service principal secret
- `AZURE_KEY_VAULT_URL` — Key Vault URL holding `AZURE-STORAGE-CONNECTION-STRING`


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.keyvault.secrets import SecretClient

# Load service-principal credentials + Key Vault URL from .env.
# load_dotenv()

# client_id     = os.environ["AZURE_CLIENT_ID"]
# tenant_id     = os.environ["AZURE_TENANT_ID"]
# client_secret = os.environ["AZURE_CLIENT_SECRET"]
# vault_url     = os.environ["AZURE_KEY_VAULT_URL"]
#Do from colab
from google.colab import userdata
client_id = userdata.get('AZURE_CLIENT_ID')
tenant_id = userdata.get('AZURE_TENANT_ID')
client_secret = userdata.get('AZURE_CLIENT_SECRET')
vault_url = userdata.get('AZURE_KEY_VAULT_URL')


# Authenticate to Azure as a service principal, then pull the storage
# connection string out of Key Vault (never hardcoded in the notebook).
credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)
secret_client = SecretClient(vault_url=vault_url, credential=credential)
connection_string = secret_client.get_secret("AZURE-STORAGE-CONNECTION-STRING").value

# Destination container created in Task 1.
container_name = "librispeech-raw"

pipeline = LibriSpeechPipeline()

# All five LibriSpeech splits we want staged in Blob Storage.
splits = ['train-clean-100', 'train-clean-360', 'train-other-500', 'dev-clean', 'test-clean']

# Process splits sequentially so we never exceed Colab disk while one is in flight.
for split in splits:
    pipeline.run(split, connection_string, container_name)


## RNN Revision

## So you know CNN — it looks at a fixed window, extracts patterns, doesn't care about sequence order. A spectrogram patch at frame 50 is processed the same way as frame 200.
## But speech is sequential. What you said 3 frames ago matters for what you're saying now. CNN has no memory of previous frames.
## RNN solves this. At every time step, the RNN takes two things as input — the current frame, and a hidden state from the previous time step. That hidden state is essentially a compressed memory of everything seen so far.


## Mostly RNN was done ; a proper notebook for the same was made attached in learning

In [3]:
import numpy as np

# let's say our input is size 4, hidden state is size 3
input_size = 4
hidden_size = 3

# weight matrix for input
W_x = np.random.randn(hidden_size, input_size)

# weight matrix for hidden state
W_h = np.random.randn(hidden_size, hidden_size)

# bias
b = np.zeros(hidden_size)

print("W_x shape:", W_x.shape)
print("W_h shape:", W_h.shape)

W_x shape: (3, 4)
W_h shape: (3, 3)


In [4]:
# one input frame (size 4)
x = np.random.randn(input_size)

# initial hidden state (all zeros at the start)
h = np.zeros(hidden_size)

print("x shape:", x.shape)
print("h shape:", h.shape)

x shape: (4,)
h shape: (3,)


In [5]:
h_new = np.tanh(W_x @ x + W_h @ h + b)

print("h_new:", h_new)
print("h_new shape:", h_new.shape)

h_new: [ 0.03987993  0.99013176 -0.92838885]
h_new shape: (3,)


In [6]:
# now move to next time step
# h_new from previous step becomes h for this step
h = h_new

# new input frame
x = np.random.randn(input_size)

# forward pass again
h_new = np.tanh(W_x @ x + W_h @ h + b)

print("h_new step 2:", h_new)

h_new step 2: [ 0.98603574 -0.6629792   0.89161399]


## LSTM

In [7]:
import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)

In [8]:
print(lstm)

LSTM(10, 20, batch_first=True)


input_size=10 means at every timestep, you're feeding in a vector of size 10. In our case later this will be mel spectrogram features.


hidden_size=20 means the LSTM's memory — both the hidden state and the cell state — will be vectors of size 20.


batch_first=True just means our input tensor will be shaped as batch, timesteps, features. We'll see this concretely in a second.

Cell state — this is the actual long term memory. Think of it like a conveyor belt. Information gets added or removed from it, but it flows through relatively undisturbed. This is what solves the vanishing gradient problem.


Hidden state — this is the short term memory. This is what the LSTM actually outputs at each timestep. It's a filtered version of the cell state.


So the analogy is simple — cell state is like your long term memory, hidden state is like what's currently on your mind right now.


In a vanilla RNN there was only a hidden state. No separate long term memory. That's why it forgot things.


LSTM added the cell state specifically to carry information across long sequences without it getting wiped out.

So LSTM = Long Short Term Memory = a network that has both a long term and short term memory running together.

In [9]:
x = torch.randn(1, 5, 10)
print(x.shape)

torch.Size([1, 5, 10])



batch_first=True means the shape is batch, timesteps, features.


So:
1 — one audio sample (batch size 1)

5 — 5 timesteps. Like 5 frames of a mel spectrogram coming in one by one.


10 — at each timestep, the feature vector is size 10. This matches our input_size=10.


So this tensor is basically saying — I have 1 audio clip, it has 5 frames, each frame has 10 features.

Why LSTM?

Vanilla RNN has only one hidden state. As the sequence gets long, that hidden state gets diluted and information from early timesteps is lost. This is the vanishing gradient problem.

LSTM Solution — Two memory streams:

Cell state = long term memory. Flows through time like a conveyor belt. Gets added to or removed from, not overwritten.

Hidden state = short term memory. What's active right now. This is the actual output at each timestep.

This is literally where the name comes from — Long Short Term Memory.

Three gates — decisions at every timestep:

Forget gate — what to throw away from cell state

Input gate — what new information to write into cell state

Output gate — what to expose as hidden state right now

Input tensor shape [1, 5, 10]:

1 = batch size (one audio clip)

5 = timesteps (5 frames of mel spectrogram)

10 = features per frame (10 mel frequency bins)

Mel frame:

One vertical slice of the mel spectrogram. Each number = energy at that frequency at that moment. High number = frequency is active. Low number = frequency is absent.

In [10]:
import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)

x = torch.randn(1, 5, 10)
print("Input shape:", x.shape)

output, (h_n, c_n) = lstm(x)
print("Output shape:", output.shape)
print("Hidden state shape:", h_n.shape)
print("Cell state shape:", c_n.shape)

Input shape: torch.Size([1, 5, 10])
Output shape: torch.Size([1, 5, 20])
Hidden state shape: torch.Size([1, 1, 20])
Cell state shape: torch.Size([1, 1, 20])


In [11]:
import torch
import torch.nn as nn

# One simple RNN cell - same weights applied at every timestep
rnn = nn.RNN(input_size=10, hidden_size=20, batch_first=True)

# 1 audio clip, 5 frames, 10 mel features per frame
x = torch.randn(1, 5, 10)

output, h_n = rnn(x)
print("Output shape:", output.shape)  # hidden state at every timestep
print("Final hidden state:", h_n.shape)  # hidden state at last timestep

Output shape: torch.Size([1, 5, 20])
Final hidden state: torch.Size([1, 1, 20])


In [12]:
import torch
import torch.nn as nn

# One RNN cell - processes ONE timestep at a time
cell = nn.RNNCell(input_size=10, hidden_size=20)

# One frame of audio - 10 mel features
frame_1 = torch.randn(1, 10)

# Initial memory - nothing remembered yet
h = torch.zeros(1, 20)

# Process frame 1
h = cell(frame_1, h)
print("Memory after frame 1:", h.shape)

Memory after frame 1: torch.Size([1, 20])


In [13]:
# Second frame of audio
frame_2 = torch.randn(1, 10)

# Pass frame 2 AND the memory from frame 1
h = cell(frame_2, h)
print("Memory after frame 2:", h.shape)

Memory after frame 2: torch.Size([1, 20])


In [14]:
# 5 frames of audio
x = torch.randn(5, 10)  # 5 frames, 10 features each

# Start with empty memory
h = torch.zeros(1, 20)

# Process one frame at a time
for i, frame in enumerate(x):
    frame = frame.unsqueeze(0)  # add batch dimension
    h = cell(frame, h)
    print(f"After frame {i+1}, memory shape: {h.shape}")

After frame 1, memory shape: torch.Size([1, 20])
After frame 2, memory shape: torch.Size([1, 20])
After frame 3, memory shape: torch.Size([1, 20])
After frame 4, memory shape: torch.Size([1, 20])
After frame 5, memory shape: torch.Size([1, 20])


In [15]:
# Let's track how much frame 1 influences later memory
x = torch.randn(5, 10)
h = torch.zeros(1, 20)

# Remember the memory after frame 1
h = cell(x[0].unsqueeze(0), h)
memory_after_frame1 = h.clone()

# Keep processing
for i in range(1, 5):
    h = cell(x[i].unsqueeze(0), h)

# How different is final memory from frame 1 memory?
difference = (h - memory_after_frame1).abs().mean().item()
print("How much memory changed from frame 1 to end:", difference)

How much memory changed from frame 1 to end: 0.4358535706996918


In [16]:
lstm_cell = nn.LSTMCell(input_size=10, hidden_size=20)

x = torch.randn(5, 10)

# LSTM has TWO memory vectors - hidden state and cell state
h = torch.zeros(1, 20)  # short term memory
c = torch.zeros(1, 20)  # long term memory

for i, frame in enumerate(x):
    frame = frame.unsqueeze(0)
    h, c = lstm_cell(frame, (h, c))
    print(f"After frame {i+1} - hidden: {h.shape}, cell: {c.shape}")

After frame 1 - hidden: torch.Size([1, 20]), cell: torch.Size([1, 20])
After frame 2 - hidden: torch.Size([1, 20]), cell: torch.Size([1, 20])
After frame 3 - hidden: torch.Size([1, 20]), cell: torch.Size([1, 20])
After frame 4 - hidden: torch.Size([1, 20]), cell: torch.Size([1, 20])
After frame 5 - hidden: torch.Size([1, 20]), cell: torch.Size([1, 20])
